# Suite RFL — Reflection

A schema is an object of its kind's meta-schema, the schema of its module form: a reference object identified by
itself, which writes itself through `accept` as a module writes it, with its name. `Reflection.of(store)` is a store
whose objects are the schemas a store registers and the named schemas they refer to, by kind, in name order; the store's
data is not read. Predicates match schemas there (mbse-patterns), and compare what they read deeply.

In [ ]:
import { Modules, Plain, Proxies, Reflection as R, Schemas as S, Validators as V } from "@mbse/schemas/Framework";
import { assert, equal, map, raises, same, text } from "./support.js";

const Phone = new S.OfObject.Builder().name("Phone").properties(text("number")).create();
const Listing = new S.OfRelation.Builder().name("Listing").links("directory", "contact").properties(text("key")).unique(
  "contact").create();
const Contact = new S.OfObject.Builder().name("Contact").ref().description("A person").properties(
  text("name"), (p) => p.name("home").of(Phone).description("Where to call"),
  (p) => p.name("tags").of((t) => t.as_indexed((i) => i.of((x) => x.as_native(String)).extent({ minimum: 0n, maximum: 9n })))).relations(
  (a) => a.name("listings").of(Listing).me("contact")).create();
const n = new S.Form.Data("variable", { name: "n" });
const Word = new S.OfNative.Builder().name("Word").type(BigInt).parameters((p) => p.name("n").of((t) => t.as_native(BigInt))).bits(n).create();
const Reach = new S.OfUnion.Builder().name("Reach").branches((b) => b.name("phone").of(Phone), (b) => b.name("other").of(Phone)).create();
const Both = new S.OfIntersection.Builder().name("Both").parts((b) => b.name("a").of(Phone), (b) => b.name("b").of(Phone)).create();
const Byte = new S.OfApply.Builder().name("Byte").of(Word).arguments(8n).create();
const Keyed = new S.OfIndexed.Builder().name("Keyed").key((t) => t.as_native(String)).of(Byte).create();
const Pair = new S.OfRelation.Builder().name("Pair").links("a", "b").create(); // no uniques
const Bare = new S.OfApply.Builder().name("Bare").of(Word).create(); // no arguments
const Loose = new S.OfObject.Builder().name("Loose").parameters((p) => p.name("x")).create(); // a parameter of any type
const Node = new S.OfObject.Builder().name("Node").properties(text("label")).create();
new S.OfObject.Builder(Node).properties((p) => p.name("next").of((t) => t.as_indexed((i) => i.of(Node)))).update();
const Twin = new S.OfObject.Builder().name("Phone").create(); // another schema of the same name
const Holder = new S.OfObject.Builder().name("Holder").properties(
  ...[Word, Reach, Both, Byte, Keyed, Bare, Loose, Node].map((k) => (p: any) => p.name((k.name as string).toLowerCase()).of(k)),
  (p) => p.name("twin").of(Twin)).create();

/** A store of proxies that registers `schemas`. */
function stocked(...schemas: any[]): Proxies.OfStore {
  const store = new Proxies.OfStore();
  for (const schema of schemas) store.register(schema);
  return store;
}

/** What a value writes through the visitor protocols, as plain data. */
function plain(value: any): any {
  if (value instanceof V.ListRecord) return value.values.map(plain);
  if (value !== null && typeof value === "object" && typeof value.accept === "function") {
    return new Map([...V.properties_of(value)].map(([name, v]) => [name, plain(v)]));
  }
  return value;
}

/** A schema's definition in a module, with its name. */
function module_form(schema: any): Map<string, unknown> {
  const modules = new Proxies.OfStore();
  const written = Plain.ToPlain(modules)(S.Module.Schema, Modules.module(modules, [schema]) as never) as any;
  const entry = written.get("objects").get("s0").get("schemas")[0];
  const [contents] = [...entry.get("schema").values()];
  return new Map([["name", entry.get("name")], ...contents]);
}

## RFL-01 · A store of schemas: what a store registers and the named schemas they refer to, by kind, in name order

In [ ]:
{
  const source = stocked(Contact, Listing);
  (source as any).Contact().name("Ada").create(); // data, which is not read
  const store = R.of(source);
  assert(same(store.extent("Schemas.Object"), [Contact, Phone]) && same(store.extent("Schemas.Relation"), [Listing])); // Phone: referred to
  assert(same(store.names(), R.META.map((meta) => meta.name)) && same(store.names(), [
    "Schemas.Native", "Schemas.Object", "Schemas.Union", "Schemas.Intersection", "Schemas.Indexed", "Schemas.Apply",
    "Schemas.Relation"]));
  assert(R.META[1] === S.OfObject.Schema && !S.OfObject.Schema.ref); // the module form's own meta-schemas
  assert(Contact.schema_name() === "Schemas.Object" && Contact.identity() === Contact.identity()
    && Contact.identity() !== Phone.identity() && Contact.owner() === null);
  const kinds = R.of(stocked(Holder, Pair));
  assert(equal(R.META.map((meta) => kinds.extent(meta.name as string).map((s: any) => s.name)), [
    ["Word"], ["Holder", "Loose", "Node", "Phone", "Phone"], ["Reach"], ["Both"], ["Keyed"], ["Bare", "Byte"], ["Pair"]]));
  assert(kinds.extent("Schemas.Object")[3] === (Twin as unknown)); // names that tie keep the order they are reached in
  assert(R.of(new Proxies.OfStore()).extent("Schemas.Object").length === 0); // a store's own meta-schemas are not its schemas
  raises(Error, () => store.extent("Contact"), "no schema registered as 'Contact'");
  source.register(Pair);
  assert(same(store.extent("Schemas.Relation"), [Listing, Pair])); // read from the registry when asked
}

## RFL-02 · Read through the visitor protocols as its module form, with its name; named schemas by name

In [ ]:
{
  const store = R.of(stocked(Contact, Listing));
  assert(equal(plain(Contact), module_form(Contact)) && equal(plain(Contact), map({
    name: "Contact", properties: [
      { name: "name", type: { native: { format: "basic", token: "str" } } },
      { name: "home", type: { named: { name: "Phone" } }, description: "Where to call" },
      { name: "tags", type: { indexed: { item: { native: { format: "basic", token: "str" } },
                                          extent: { minimum: 0n, maximum: 9n } } } }],
    adjacencies: [{ name: "listings", relation: { named: { name: "Listing" } }, me: "contact" }],
    ref: true, description: "A person" })));
  assert([Phone, Listing, Word, Reach, Both, Byte, Keyed, Pair, Bare, Loose, Node, Holder].every((schema) =>
    equal(plain(schema), module_form(schema))));
  assert(!plain((Contact.properties.get("name") as S.OfProperty.Data).type).has("name")); // an unnamed schema
  assert(store.member(Contact, "name") === "Contact" && store.member(Contact, "singleton") === null);
}

## RFL-03 · Validated as their meta-schemas; another format's token; an unsupported token; a store of schemas builds nothing

In [ ]:
{
  const kinds = R.of(stocked(Holder, Pair));
  assert(R.META.every((meta) => kinds.extent(meta.name as string).every((schema) => V.Validate(kinds)(meta, schema as never).length === 0)));
  const Size = S.OfNative.resolve((t) => t.name("Size").token("ccpp", "size_t"));
  const sized = R.of(stocked(new S.OfObject.Builder().name("Sized").properties((p) => p.name("n").of(Size)).create()));
  assert(same(sized.extent("Schemas.Native"), [Size]) && equal(plain(Size), map({ name: "Size", format: "ccpp", token: "size_t" })));
  raises(TypeError, () => V.properties_of(new S.OfNative.Data("x" as never) as never), "unsupported native type 'x'");
  raises(TypeError, () => kinds.builder("Schemas.Object"), "schemas are built by their builders, not by a store of schemas");
}